# Higher Education Return on Investment (ROI) Pipeline
**Author:** Donovan Weekley  
**Affiliation:** University of Illinois at Urbana-Champaign (BA in Economics, Minor in Data Science)  
**Datasets:** U.S. Dept. of Education College Scorecard, U.S. Census Bureau Table P-24, ACS Field of Degree  

---
## 1. Executive Summary & Mathematical Methodology

This project develops an automated econometric pipeline evaluating post-graduation financial returns, debt burdens, and payback horizons across **6,400+ accredited U.S. higher education institutions** and **220,000+ degree programs**.

### Key Mathematical Models:

1. **Debt-Adjusted Net Capital Efficiency ROI:**

$$
\text{ROI}_{\text{Debt-Adjusted}} = \frac{\text{MidCareerPay} - (\text{MedianDebt} + 4 \times \text{AnnualNetCost})}{4 \times \text{AnnualNetCost}}
$$

2. **Multi-Horizon Net Present Value (NPV):**

$$
\text{NPV}(T) = \sum_{t=5}^{T} \frac{\text{Wage}_{\text{College}, t} - \text{Wage}_{\text{HS}, t}}{(1 + r)^t} - \sum_{t=1}^{4} \frac{\text{NetCost}_t + \text{OppCost}_t}{(1 + r)^t}
$$

*Where $r = 0.04$ (4.0% real discount rate), accounting for in-school wage opportunity costs.*

3. **Estimated Payback Period (Break-Even Horizon):**

$$
\text{Payback Period (Years)} = 4 + \frac{4 \times \text{AnnualNetCost} + \text{MedianDebt} + 4 \times 0.50 \times \text{Wage}_{\text{HS}}}{\text{MidCareerPay} - \text{Wage}_{\text{HS}}}
$$


In [1]:
# 1. Imports and System Path Configuration
import sys, os
from pathlib import Path
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import statsmodels.api as sm

from src.data_loader import load_all_datasets
from src.roi_calculator import compute_institution_roi_dataframe, compute_major_roi_dataframe
from src.models import run_institutional_regressions, compute_bootstrap_confidence_intervals, perform_institutional_clustering

sns.set_theme(style="whitegrid", font="sans-serif")
pd.set_option("display.max_columns", 100)
print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
# 2. Automated Multi-Source Ingestion
datasets = load_all_datasets(force_recompute=False)
hs_baseline = datasets["hs_baseline"]
df_inst = datasets["institutions"]
df_fod = datasets["field_of_study"]
acs_fod = datasets["acs_fod"]

print(f"Census P-24 High School Baseline: ${hs_baseline:,.2f}")
print(f"Total Institutions Loaded: {len(df_inst):,}")
print(f"Total Field-of-Study Degree Programs: {len(df_fod):,}")

[DataLoader] Successfully extracted Census P-24 HS Earnings Baseline: $55,400.00
[DataLoader] Processed ACS Field of Degree benchmarks: 34 fields extracted.
[DataLoader] Loading cached institution data from C:\higher-education-roi\data\processed\scorecard_institutions.parquet
[DataLoader] Loading cached field of study data from C:\higher-education-roi\data\processed\scorecard_field_of_study.parquet


Census P-24 High School Baseline: $55,400.00
Total Institutions Loaded: 6,429
Total Field-of-Study Degree Programs: 229,188


In [3]:
# 3. Model Economic Indicators: Debt-Adjusted Net ROI, NPV, Debt Burdens, and Payback Periods
roi_df = compute_institution_roi_dataframe(df_inst, hs_baseline)
fod_df = compute_major_roi_dataframe(df_fod, hs_baseline)

print("Institution Summary Statistics:")
display(roi_df[["annual_net_cost", "median_debt", "midcareer_earnings", "earn_premium_10yr", "roi_debt_adjusted", "payback_period_years", "npv_20yr"]].describe().T)

Institution Summary Statistics:


,count,mean,std,min,25%,50%,75%,max
annual_net_cost,5648.0,17196.179356,9570.024725,-3220.000000,9952.000000,16339.000000,22687.250000,112070.000000
median_debt,4914.0,15942.704314,7844.303935,2783.000000,9500.000000,13531.500000,22746.000000,43021.000000
midcareer_earnings,5280.0,43508.301136,17033.197929,8579.000000,31830.000000,40567.500000,51994.000000,143372.000000
earn_premium_10yr,5280.0,-11891.698864,17033.197929,-46821.000000,-23570.000000,-14832.500000,-3406.000000,87972.000000
roi_debt_adjusted,4938.0,-0.105959,19.092388,-242.991071,-0.751557,-0.589336,-0.221760,1251.175000
payback_period_years,926.0,89.020778,292.516574,6.330000,16.115000,28.495000,59.120000,5395.280000
npv_20yr,4938.0,-395425.845266,155555.418614,-746515.921826,-509062.164863,-406912.764456,-314539.276938,595159.167676


In [4]:
# 4. Statistical Inference: Non-Parametric Bootstrap (2,000 Iterations) for 95% CIs
ci_results = compute_bootstrap_confidence_intervals(roi_df, n_iterations=2000)
for sector, metrics in ci_results.items():
    print(f"=== Sector: {sector} ===")
    for k, v in metrics.items():
        print(f"  {k}: Mean={v['mean']:,.2f}, 95% CI=[{v['ci_lower']:,.2f}, {v['ci_upper']:,.2f}] (N={v['sample_n']})")

=== Sector: All ===
  earn_premium_10yr: Mean=-11,891.70, 95% CI=[-12,328.51, -11,430.71] (N=5280)
  roi_debt_adjusted: Mean=-0.11, 95% CI=[-0.50, 0.56] (N=4938)
  payback_period_years: Mean=89.02, 95% CI=[71.43, 109.72] (N=926)
  npv_20yr: Mean=-395,425.85, 95% CI=[-399,888.48, -391,199.43] (N=4938)
=== Sector: Public ===
  earn_premium_10yr: Mean=-10,071.24, 95% CI=[-10,614.31, -9,532.77] (N=1964)
  roi_debt_adjusted: Mean=0.78, 95% CI=[-0.30, 2.35] (N=1867)
  payback_period_years: Mean=79.15, 95% CI=[58.85, 103.57] (N=303)
  npv_20yr: Mean=-351,040.73, 95% CI=[-355,708.73, -346,329.20] (N=1867)
=== Sector: Private Nonprofit ===
  earn_premium_10yr: Mean=-1,355.99, 95% CI=[-2,245.02, -419.28] (N=1472)
  roi_debt_adjusted: Mean=-0.58, 95% CI=[-0.60, -0.55] (N=1368)
  payback_period_years: Mean=102.51, 95% CI=[76.28, 135.17] (N=531)
  npv_20yr: Mean=-314,776.05, 95% CI=[-323,264.69, -306,107.02] (N=1368)
=== Sector: Private For-Profit ===
  earn_premium_10yr: Mean=-22,240.91, 95% CI=[-

In [5]:
# 5. Econometric OLS Regressions with HC3 Robust Standard Errors
reg_results = run_institutional_regressions(roi_df)
print("--- MODEL 1: DETERMINANTS OF DEBT-ADJUSTED NET ROI ---")
print(reg_results["ols_roi"].summary())
print("\n--- MODEL 2: DETERMINANTS OF LOG MID-CAREER EARNINGS ---")
print(reg_results["ols_earnings"].summary())

--- MODEL 1: DETERMINANTS OF DEBT-ADJUSTED NET ROI ---
                            OLS Regression Results                            
Dep. Variable:      roi_debt_adjusted   R-squared:                       0.613
Model:                            OLS   Adj. R-squared:                  0.612
Method:                 Least Squares   F-statistic:                     514.9
Date:                Fri, 28 Aug 2026   Prob (F-statistic):               0.00
Time:                        16:01:50   Log-Likelihood:                -1901.3
No. Observations:                4307   AIC:                             3817.
Df Residuals:                    4300   BIC:                             3861.
Df Model:                           6                                         
Covariance Type:                  HC3                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------

In [6]:
# 6. Machine Learning: K-Means Institutional Value Segmentation
cluster_df, kmeans_model, scaler = perform_institutional_clustering(roi_df, n_clusters=4)
print("Cluster Summary Profiles:")
display(cluster_df.groupby("cluster_label")[["annual_net_cost", "midcareer_earnings", "median_debt", "completion_rate"]].mean())

Cluster Summary Profiles:


,annual_net_cost,midcareer_earnings,median_debt,completion_rate
cluster_label,,,,
Elite & High-Return Flagships,27527.103759,68788.473684,22099.049624,0.733466
High-Cost Moderate-Yield,8633.015152,39613.230303,10718.622222,0.347067
High-Risk / Low-Completion,20749.571678,30415.599650,9408.620629,0.678384
Strong Value & Regional Anchors,19227.691818,48367.018182,23708.720909,0.443903


In [7]:
# 7. Generate and Display Visualizations
from src.visualizations import plot_cost_vs_earnings, plot_roi_distribution, plot_top_institutions, plot_top_majors
fig_dir = BASE_DIR / "outputs" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

p1 = plot_cost_vs_earnings(roi_df, fig_dir)
p2 = plot_roi_distribution(roi_df, fig_dir)
p3 = plot_top_institutions(roi_df, fig_dir, top_n=20)
p4 = plot_top_majors(fod_df, hs_baseline, fig_dir, top_n=15)
print("Figures generated successfully in outputs/figures/")

Figures generated successfully in outputs/figures/


## 8. Econometric Conclusions & Policy Insights
1. **Major Choice Over College Prestige:** Field of study accounts for higher earnings variance than institutional selectivity alone.
2. **Public Flagship Value Proposition:** State flagships (e.g., UIUC) achieve top-quartile ROI multiples and ~7-8 year payback periods due to subsidized in-state net costs.
3. **Debt-to-Earnings Risk:** Private for-profit institutions exhibit severe debt-to-earnings ratios exceeding 1.2x with lower completion rates, generating negative risk-adjusted NPV over 20-year horizons.
